# Order Book Signals on Bond & Equity Calendar Spreads

## 1  Setup & Connection

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')

import polars as pl
import polars.selectors as cs
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from plotnine import *
from plotnine.themes import theme_bw

from src.utils import (
    connect_snowflake,
    create_snowpark_session,
    retrieve_polars_from_snowpark,
    unpack_kwargs,
    unpack_kwargs_for_agg,
    read_table,
    parse_security,
    build_contract_calendar,
    add_roll_window,
    add_microstructure_signals,
    acf_by_security,
    ljungbox_by_security,
    newey_west_maxlags,
)
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
from functools import reduce
import operator

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(20)
print('polars', pl.__version__)

polars 1.41.2


In [2]:
DB     = 'LISTED_INTERN_PROJECT'
SCHEMA = 'PROJECT_5'
TABLES = ['BINNED_DATA', 'QCODE_MAPPING', 'SECURITY_META']

snowflake_conn = connect_snowflake('../.env')
snowpark = create_snowpark_session('../.env')
print('Connected.')

Connected.


## 2  Data Ingestion

Pull only the required columns from each Snowflake table (the projection is pushed down to
Snowflake so we transfer the minimum over the wire). Column names come back lower-cased.

In [3]:
# --- 2  Data Ingestion -------------------------------------------------------------
# Small-scale subset: 2 Phys + 2 Cash qcodes, years 2024-2025.
#
# Futures and calendar spreads of the same product carry DIFFERENT QCODEs in BINNED_DATA,
# so we cannot use a single QCODE filter to capture both instrument types.  Instead:
#   • Futures  — matched by QCODE AND SECURITY NOT LIKE '%/%' (no slash in name).
#   • Spreads  — matched by SECURITY LIKE pattern: {BBG_CODE}%/% {YELLOW_KEY}.
#
# The 2024-2025 date restriction is applied in Section 3 after the contract calendar
# is joined (target_date is delivery-type-aware and not available until that join).
#
# Note: Snowpark's Column.contains() treats its argument as a column identifier, not a
# literal string. All substring checks are expressed as LIKE patterns to avoid this.

BINNED_COLS = [
    'QCODE', 'SECURITY', 'BIN_START_TIME', 'PUBLICATION_DATE',
    'BID_SIZE_START', 'ASK_SIZE_START', 'BID_START', 'ASK_START',
    'VOLUME', 'SIGNED_VOLUME',
]
QCODE_COLS    = ['QCODE', 'BBG_CODE', 'YELLOW_KEY', 'DELIVERY', 'IS_CONVENTION_BUY_NEAR']
SEC_META_COLS = ['SECURITY', 'LAST_TRADE_DATE', 'FIRST_NOTICE_DATE']

qmap     = read_table(snowpark, DB, SCHEMA, 'QCODE_MAPPING', QCODE_COLS)
sec_meta = read_table(snowpark, DB, SCHEMA, 'SECURITY_META', SEC_META_COLS)

# Pick 2 Phys + 2 Cash qcodes for small-scale workshopping.
phys_qcodes = qmap.filter(pl.col('delivery') == 'Phys')['qcode'].unique().to_list()[:2]
cash_qcodes = qmap.filter(pl.col('delivery') == 'Cash')['qcode'].unique().to_list()[:2]
subset_qcodes = phys_qcodes + cash_qcodes
print(f'Phys qcodes : {phys_qcodes}')
print(f'Cash qcodes : {cash_qcodes}')

# BBG_CODE + YELLOW_KEY for the 4 selected products — used to construct the spread filter.
subset_products = (
    qmap.filter(pl.col('qcode').is_in(subset_qcodes))
    .select('bbg_code', 'yellow_key')
    .unique()
)

fqn_binned = f'{DB}.{SCHEMA}.BINNED_DATA'

# Futures: QCODE in our subset AND SECURITY has no slash (LIKE '%/%' negated).
futures_filter = (
    F.col('QCODE').isin(subset_qcodes) &
    ~F.col('SECURITY').like('%/%')
)

# Spreads: SECURITY matches pattern  {BBG_CODE}.../{...} {YELLOW_KEY}.
# Using Snowflake LIKE: '%' = zero-or-more wildcard characters.

spread_clauses = [
    F.col('SECURITY').like(f'{row["bbg_code"]}%/% {row["yellow_key"]}')
    for row in subset_products.iter_rows(named=True)
]
spread_filter = reduce(operator.or_, spread_clauses)

snow_binned = (
    snowpark.table(fqn_binned)
    .select(*BINNED_COLS)
    .filter(futures_filter | spread_filter)
)
binned = retrieve_polars_from_snowpark(snow_binned)

n_fut = binned.filter(~pl.col('security').str.contains('/')).height
n_spr = binned.filter(pl.col('security').str.contains('/')).height
print(f'binned   : {binned.shape}  (futures: {n_fut:,}  |  spreads: {n_spr:,})')
print(f'qmap     : {qmap.shape}')
print(f'sec_meta : {sec_meta.shape}')
binned.head()

Phys qcodes : ['BF', 'GH']
Cash qcodes : ['TT', 'AZ']
binned   : (3262124, 10)  (futures: 2,202,500  |  spreads: 1,059,624)
qmap     : (18, 5)
sec_meta : (1106, 3)


qcode,security,bin_start_time,publication_date,bid_size_start,ask_size_start,bid_start,ask_start,volume,signed_volume
str,str,time,date,f64,f64,f64,f64,i32,i32
"""AZ""","""EO2017V/2017X Index""",09:05:00,2017-09-15,15.0,3.0,1.85,2.15,0,0
"""AZ""","""EO2017V/2017X Index""",12:55:00,2017-09-15,1.0,3.0,1.9,2.15,0,0
"""GH""","""DU2017H Comdty""",11:05:00,2016-10-25,3.0,99.0,112.075,112.09,0,0
"""AZ""","""EO2017V Index""",14:05:00,2017-08-29,12.0,4.0,507.95,508.05,0,0
"""AZ""","""EO2017V Index""",15:25:00,2017-08-29,3.0,14.0,507.6,507.7,0,0


## 3  Merge, Parse Tickers, Build the Contract Calendar & Filter to 2024-2025

**Step 1** — merge `BINNED_DATA` with `QCODE_MAPPING` on `QCODE`. Spreads pulled via name
pattern may carry a QCODE outside the futures subset; their qmap columns are left as null and
the authoritative `delivery` comes from the calendar (Step 3).

**Step 2** — parse the `SECURITY` string to flag futures vs. calendar spreads and, for every
spread, construct the explicit **near** and **far** single-future tickers. Each row also gets a
`meta_key`: the near leg for spreads (since `SECURITY_META` has no spread rows), the contract
itself for futures.

**Step 3 & 4** — build a per-future roll calendar from `SECURITY_META`: pick the target date
(`LAST_TRADE_DATE` for `Cash`, `FIRST_NOTICE_DATE` for `Phys`), compute each contract's own
10-business-day roll window, and attach the *previous* chronological contract's window. Join the
calendar onto the data via `meta_key`. The qmap's `delivery` column (null for spreads) is dropped
before this join so the calendar's `delivery` (always resolved from BBG_CODE + YELLOW_KEY) is used.

**Step 5** — derive the trading `date` from `PUBLICATION_DATE` (`BIN_START_TIME` is time-only).

**Step 6** — keep only rows whose near-leg `target_date` falls in **2024 or 2025**. This is the
date-range restriction for the workshop subset.

In [5]:
# --- 3  Merge + parse + calendar + date filter ------------------------------------

# Step 1: merge BINNED_DATA with QCODE_MAPPING on QCODE.
# Spreads pulled via the name-pattern filter may have QCODEs outside subset_qcodes,
# so their bbg_code / yellow_key / delivery / is_convention_buy_near will be null here.
data = binned.join(qmap, on='qcode', how='left')

# Step 2: parse SECURITY -> is_spread, near/far identifiers, meta_key.
data = parse_security(data, col='security')

data.head()

# Steps 3-4: build the per-future roll calendar (target date + own/previous roll windows).
calendar = build_contract_calendar(sec_meta, qmap, roll_days=10)

# Drop qmap's 'delivery' before joining the calendar so the calendar's 'delivery'
# (correctly resolved from BBG_CODE+YELLOW_KEY for every security) is the sole authority.
data = data.drop('delivery')
data = data.join(
    calendar.rename({'security': 'meta_key'}),
    on='meta_key',
    how='left',
)

# Step 5: trading date from PUBLICATION_DATE (BIN_START_TIME is time-only, not a date).
data = data.with_columns(date=pl.col('publication_date').cast(pl.Date))

# Step 6: keep only rows whose near-leg target date is in 2024 or 2025.
# target_date = FIRST_NOTICE_DATE (Phys) or LAST_TRADE_DATE (Cash), already set by the calendar.
data = data.filter(pl.col('publication_date').dt.year().is_in([2024, 2025]))

n_spread = data.filter(pl.col('is_spread')).height
print(f'rows: {data.height:,}  |  spread rows: {n_spread:,}  |  future rows: {data.height - n_spread:,}')
data.select(
    'security', 'is_spread', 'near_identifier', 'far_identifier', 'meta_key',
    'delivery', 'target_date', 'roll_start', 'roll_end', 'prev_roll_start', 'prev_roll_end',
).head()

rows: 631,358  |  spread rows: 207,748  |  future rows: 423,610


security,is_spread,near_identifier,far_identifier,meta_key,delivery,target_date,roll_start,roll_end,prev_roll_start,prev_roll_end
str,bool,str,str,str,str,date,date,date,date,date
"""DU2024U/2024Z Comdty""",true,"""DU2024U Comdty""","""DU2024Z Comdty""","""DU2024U Comdty""","""Phys""",2024-09-06,2024-08-23,2024-09-05,2024-05-23,2024-06-05
"""EO2024Q Index""",false,"""EO2024Q Index""",null,"""EO2024Q Index""","""Cash""",2024-08-16,2024-08-02,2024-08-15,2024-07-05,2024-07-18
"""RX2024M Comdty""",false,"""RX2024M Comdty""",null,"""RX2024M Comdty""","""Phys""",2024-06-06,2024-05-23,2024-06-05,2024-02-22,2024-03-06
"""EO2024U Index""",false,"""EO2024U Index""",null,"""EO2024U Index""","""Cash""",2024-09-20,2024-09-06,2024-09-19,2024-08-02,2024-08-15
"""EO2025G Index""",false,"""EO2025G Index""",null,"""EO2025G Index""","""Cash""",2025-02-21,2025-02-07,2025-02-20,2025-01-03,2025-01-16


## 4  Roll-Period Filtering → `df_cs` & `df_combined`

**Step 7** — keep only rows whose `date` (from `PUBLICATION_DATE`) falls inside the relevant
roll window:

- **Calendar spreads** — inside their *own* roll period (their near leg's window).
- **Futures** — inside *either* their own roll period *or* their immediately-previous
  chronological contract's roll period.

Outputs:
- `df_cs` — filtered calendar spreads only.
- `df_combined` — filtered calendar spreads **and** filtered futures.

In [6]:
# --- 5  Roll-period filtering -----------------------------------------------------

# A bin sits inside a [start, end] window (inclusive). Null bounds (missing metadata) -> False.
def _in_window(start: str, end: str) -> pl.Expr:
    return pl.col('date').is_between(pl.col(start), pl.col(end), closed='both').fill_null(False)

in_own_roll  = _in_window('roll_start', 'roll_end')
in_prev_roll = _in_window('prev_roll_start', 'prev_roll_end')

# Calendar spreads: own (near-leg) roll period only.
df_cs = data.filter(pl.col('is_spread') & in_own_roll)

# Futures: own roll period OR previous contract's roll period.
df_fut = data.filter(~pl.col('is_spread') & (in_own_roll | in_prev_roll))

# Combined: filtered spreads + filtered futures (aligned schema via vertical concat).
df_combined = pl.concat([df_cs, df_fut], how='vertical')

print(f'df_cs       : {df_cs.shape}')
print(f'df_fut      : {df_fut.shape}')
print(f'df_combined : {df_combined.shape}')
df_cs.head()

df_cs       : (50294, 27)
df_fut      : (100640, 27)
df_combined : (150934, 27)


qcode,security,bin_start_time,publication_date,bid_size_start,ask_size_start,bid_start,ask_start,volume,signed_volume,bbg_code,yellow_key,is_convention_buy_near,is_spread,near_identifier,far_identifier,near_expiry_key,far_expiry_key,meta_key,expiry_key,delivery,target_date,roll_start,roll_end,prev_roll_start,prev_roll_end,date
str,str,time,date,f64,f64,f64,f64,i32,i32,str,str,f64,bool,str,str,i32,i32,str,i32,str,date,date,date,date,date,date
"""GH""","""DU2024H/2024M Comdty""",13:10:00,2024-03-01,14412.0,5329.0,-0.525,-0.52,601,587,"""DU""","""Comdty""",1.0,true,"""DU2024H Comdty""","""DU2024M Comdty""",202403,202406,"""DU2024H Comdty""",202403,"""Phys""",2024-03-07,2024-02-22,2024-03-06,2023-11-23,2023-12-06,2024-03-01
"""AZ""","""EO2024X/2024Z Index""",13:00:00,2024-11-08,468.0,1450.0,-3.0,-2.95,0,0,"""EO""","""Index""",1.0,true,"""EO2024X Index""","""EO2024Z Index""",202411,202412,"""EO2024X Index""",202411,"""Cash""",2024-11-15,2024-11-01,2024-11-14,2024-10-04,2024-10-17,2024-11-08
"""AZ""","""EO2024K/2024M Index""",13:30:00,2024-05-09,256.0,143.0,-2.55,-2.5,0,0,"""EO""","""Index""",1.0,true,"""EO2024K Index""","""EO2024M Index""",202405,202406,"""EO2024K Index""",202405,"""Cash""",2024-05-17,2024-05-03,2024-05-16,2024-04-05,2024-04-18,2024-05-09
"""BF""","""RX2024Z/2025H Comdty""",13:10:00,2024-11-26,30766.0,14501.0,-1.53,-1.52,1,1,"""RX""","""Comdty""",1.0,true,"""RX2024Z Comdty""","""RX2025H Comdty""",202412,202503,"""RX2024Z Comdty""",202412,"""Phys""",2024-12-06,2024-11-22,2024-12-05,2024-08-23,2024-09-05,2024-11-26
"""GH""","""DU2024U/2024Z Comdty""",08:30:00,2024-09-05,2038.0,1324.0,-0.455,-0.45,0,0,"""DU""","""Comdty""",1.0,true,"""DU2024U Comdty""","""DU2024Z Comdty""",202409,202412,"""DU2024U Comdty""",202409,"""Phys""",2024-09-06,2024-08-23,2024-09-05,2024-05-23,2024-06-05,2024-09-05


## 5  Microstructure Signal Generation (Research Plan §3)

Compute the order-book / flow signals on the filtered calendar-spread set `df_cs`. **CRITICAL —
strictly intraday:** all lag-dependent quantities are partitioned by the trading **session
`[SECURITY, DATE]`** (not just the contract) and ordered by `BIN_START_TIME`, so a lag never
spans an overnight gap, a weekend, or a day boundary. The first bin of each session has null
lags and is dropped. Using the per-bin `*_START` quotes:

| Signal | Definition |
|---|---|
| $P_t$ (`mid_price`) | $(\text{BID\_START} + \text{ASK\_START})/2$ |
| $\Delta P_t$ (`delta_p`) | $P_t - P_{t-1}$ (within session) |
| $OBI_t$ (`obi`) | $\text{BID\_SIZE\_START} - \text{ASK\_SIZE\_START}$ |
| $\Delta L_t^b$ (`delta_lb`) | $Q_t^b-Q_{t-1}^b$ if $P_t^b=P_{t-1}^b$; $\;Q_t^b$ if $P_t^b>P_{t-1}^b$; $\;-Q_{t-1}^b$ if $P_t^b<P_{t-1}^b$ |
| $\Delta L_t^a$ (`delta_la`) | mirror of the bid (an ask *improvement* is a price **decrease**) |
| $OBC_t$ (`ofi`) | $\Delta L_t^b - \Delta L_t^a$  (order-flow imbalance) |
| $STV_t$ (`stv`) | `SIGNED_VOLUME` (dataset); empty bins → 0 |
| $NOI_t$ (`noi`) | $OBC_t - STV_t$ |

A `date` session key is added automatically. Result is saved to **`df_signals`**.

In [7]:
# --- 5  Signal generation (partitioned by SESSION [security, date], ordered by time) ----
df_signals = add_microstructure_signals(df_cs, security_col='security', time_col='bin_start_time')

n_sessions = df_signals.select(['security', 'date']).n_unique()
print(f'df_signals: {df_signals.shape}  ({df_signals["security"].n_unique()} spreads, {n_sessions} sessions)')
df_signals.select(
    'security', 'date', 'bin_start_time', 'mid_price', 'delta_p', 'obi',
    'delta_lb', 'delta_la', 'ofi', 'stv', 'noi',
).head()

df_signals: (49814, 35)  (48 spreads, 480 sessions)


security,date,bin_start_time,mid_price,delta_p,obi,delta_lb,delta_la,ofi,stv,noi
str,date,time,f64,f64,f64,f64,f64,f64,i32,f64
"""DU2024H/2024M Comdty""",2024-02-22,08:15:00,-0.5475,-0.005,-85.0,-130.0,173.0,-303.0,0,-303.0
"""DU2024H/2024M Comdty""",2024-02-22,08:20:00,-0.5475,0.0,847.0,932.0,0.0,932.0,0,932.0
"""DU2024H/2024M Comdty""",2024-02-22,08:25:00,-0.5475,0.0,-81.0,-928.0,0.0,-928.0,0,-928.0
"""DU2024H/2024M Comdty""",2024-02-22,08:30:00,-0.5475,0.0,-81.0,0.0,0.0,0.0,0,0.0
"""DU2024H/2024M Comdty""",2024-02-22,08:35:00,-0.5475,0.0,-81.0,0.0,0.0,0.0,0,0.0


## 6  Hypothesis H1 — Autocorrelation & Persistence (Research Plan §5.1)

**H1:** both Signed Trade Volume ($STV_t$) and Net Order Inflow ($NOI_t$) exhibit significant,
slowly-decaying positive autocorrelation.

To keep the analysis **strictly intraday**, the ACF and Ljung-Box Q-test are computed
**per session `[SECURITY, DATE]`** (ordered by `BIN_START_TIME`) and then aggregated —
concatenating across day boundaries would inject spurious overnight/weekend lags. For each
$X \in \{STV_t, NOI_t\}$ we report the cross-sectional **mean ACF** out to lag $L=20$ (with the
$\pm 1.96/\sqrt{\bar n}$ white-noise band) and the **Ljung-Box** Q-test, validating H1 if the
null of no serial correlation is rejected at $p < 0.01$.

In [ ]:
# --- 6a  Sample ACF up to lag L = 12 (per SESSION [security, date], then averaged) ----
L = 12
SESSION = ['security', 'date']
acf_results = {}

# Mean session length (used for the approximate white-noise band).
avg_n = df_signals.group_by(SESSION).len()['len'].mean()
band = 1.96 / np.sqrt(avg_n)

for col in ['stv', 'noi', 'obi', 'ofi']:
    lags, mean_acf, per_sec, n_series = acf_by_security(
        df_signals, value_col=col, group_cols=SESSION,
        time_col='bin_start_time', nlags=L, min_obs=50,
    )
    acf_results[col] = mean_acf
    print(f'\n=== ACF of {col.upper()}  ({n_series} sessions, ~{avg_n:,.0f} bins each) ===')
    print(f'    95% white-noise band ~ +/-{band:.3f}')
    for lag, val in zip(lags, mean_acf):
        flag = '*' if abs(val) > band else ' '
        print(f'    lag {lag:>2}: {val:+.4f} {flag}')

acf_table = pl.DataFrame({'lag': list(range(1, L + 1)),
                          'acf_stv': acf_results['stv'],
                          'acf_noi': acf_results['noi'],
                          'acf_obi': acf_results['obi'],
                          'acf_ofi': acf_results['ofi']})
acf_table


=== ACF of STV  (470 sessions, ~104 bins each) ===
    95% white-noise band ~ +/-0.192
    lag  1: +0.0112  
    lag  2: +0.0069  
    lag  3: +0.0008  
    lag  4: -0.0029  
    lag  5: +0.0063  
    lag  6: -0.0018  
    lag  7: +0.0015  
    lag  8: -0.0043  
    lag  9: -0.0051  
    lag 10: -0.0072  
    lag 11: -0.0065  
    lag 12: -0.0031  

=== ACF of NOI  (479 sessions, ~104 bins each) ===
    95% white-noise band ~ +/-0.192
    lag  1: -0.1301  
    lag  2: -0.0100  
    lag  3: +0.0067  
    lag  4: -0.0108  
    lag  5: +0.0024  
    lag  6: -0.0026  
    lag  7: -0.0065  
    lag  8: -0.0033  
    lag  9: -0.0037  
    lag 10: -0.0052  
    lag 11: -0.0132  
    lag 12: -0.0022  

=== ACF of OBI  (478 sessions, ~104 bins each) ===
    95% white-noise band ~ +/-0.192
    lag  1: +0.7518 *
    lag  2: +0.6261 *
    lag  3: +0.5309 *
    lag  4: +0.4583 *
    lag  5: +0.3974 *
    lag  6: +0.3485 *
    lag  7: +0.3019 *
    lag  8: +0.2619 *
    lag  9: +0.2309 *
    lag 10

lag,acf_stv,acf_noi,acf_obi,acf_ofi
i64,f64,f64,f64,f64
1,0.011171,-0.13009,0.751837,0.001461
2,0.006928,-0.010038,0.626132,-0.008126
3,0.000825,0.006737,0.530887,0.003461
4,-0.002889,-0.010828,0.458295,-0.007534
5,0.006321,0.002362,0.397399,-0.001939
6,-0.001774,-0.00257,0.348495,0.000429
7,0.001547,-0.006506,0.301853,-0.00594
8,-0.004278,-0.003265,0.261908,-0.004892
9,-0.005147,-0.003718,0.230892,-0.006057


In [10]:
# --- 6b  Ljung-Box Q-test (cumulative through lag 20), per SESSION [security, date] ----
ALPHA = 0.01

for col in ['stv', 'noi', 'obi', 'ofi']:
    lb = ljungbox_by_security(
        df_signals, value_col=col, group_cols=SESSION,
        time_col='bin_start_time', lag=L, min_obs=50,
    )
    n_total = lb.height
    n_reject = lb.filter(pl.col('lb_pvalue') < ALPHA).height
    print(f'\n=== Ljung-Box Q-test on {col.upper()} (lag {L}) ===')
    print(f'    sessions tested       : {n_total}')
    print(f'    reject H0 @ p<{ALPHA}    : {n_reject}/{n_total} ({n_reject / n_total:.1%})')
    print(f'    median Q p-value      : {lb["lb_pvalue"].median():.2e}')
    print(f'    median Q-statistic    : {lb["lb_stat"].median():,.1f}')

# --- H1 verdict --------------------------------------------------------------------
print('\n' + '=' * 60)
print('H1 verdict: significant, slowly-decaying POSITIVE autocorrelation')
print('is confirmed when the mean ACF stays positive above the band across')
print(f'lags and the Ljung-Box null is rejected (p<{ALPHA}) for ~all sessions.')


=== Ljung-Box Q-test on STV (lag 12) ===
    sessions tested       : 470
    reject H0 @ p<0.01    : 31/470 (6.6%)
    median Q p-value      : 7.93e-01
    median Q-statistic    : 7.9

=== Ljung-Box Q-test on NOI (lag 12) ===
    sessions tested       : 479
    reject H0 @ p<0.01    : 92/479 (19.2%)
    median Q p-value      : 2.21e-01
    median Q-statistic    : 15.4

=== Ljung-Box Q-test on OBI (lag 12) ===
    sessions tested       : 478
    reject H0 @ p<0.01    : 463/478 (96.9%)
    median Q p-value      : 7.18e-43
    median Q-statistic    : 232.3

=== Ljung-Box Q-test on OFI (lag 12) ===
    sessions tested       : 479
    reject H0 @ p<0.01    : 48/479 (10.0%)
    median Q p-value      : 4.71e-01
    median Q-statistic    : 11.7

H1 verdict: significant, slowly-decaying POSITIVE autocorrelation
is confirmed when the mean ACF stays positive above the band across
lags and the Ljung-Box null is rejected (p<0.01) for ~all sessions.


## 7  Hypothesis H2 — Contemporaneous Price Impact (Research Plan §5.2)

Fit the contemporaneous regression

$$\Delta P_t = \beta_0 + \beta_1 STV_t + \beta_2 NOI_t + \beta_3 OBI_t + \epsilon_t$$

$OFI_t$ (`ofi`) is **omitted** because $NOI_t = OFI_t - STV_t$ makes the three flow metrics
perfectly collinear. The regression is contemporaneous (no lags), so all spreads are pooled;
the data stays sorted by `[security, date, bin_start_time]` so the Newey-West HAC window operates
mostly *within* an intraday session. We use **Newey-West HAC** standard errors with lag truncation
$m = \lfloor 4 (T/100)^{2/9} \rfloor$.

**H2 is validated** if $\beta_1$ (STV) and $\beta_2$ (NOI) are positive and highly significant
($t > 2.5$), while $\beta_3$ (OBI) is near zero / insignificant.

In [11]:
# --- 7a  Fit the contemporaneous model with Newey-West HAC errors -----------------
# y = delta_p ; X = [STV, NOI, OBI] (+ const). OFI/ofi omitted to avoid perfect collinearity.
reg_cols = ['delta_p', 'stv', 'noi', 'obi']
reg_df = df_signals.select(reg_cols).drop_nulls().to_pandas()

T = len(reg_df)                       # total valid observations
m = newey_west_maxlags(T)             # m = floor(4 * (T/100)**(2/9))
print(f'Valid observations T = {T:,}')
print(f'Newey-West lag truncation m = {m}')

y = reg_df['delta_p']
X = sm.add_constant(reg_df[['stv', 'noi', 'obi']])

model_h2 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': m})
print(model_h2.summary())

Valid observations T = 49,814
Newey-West lag truncation m = 15
                            OLS Regression Results                            
Dep. Variable:                delta_p   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     50.90
Date:                Mon, 08 Jun 2026   Prob (F-statistic):           7.69e-33
Time:                        11:12:08   Log-Likelihood:             1.3046e+05
No. Observations:               49814   AIC:                        -2.609e+05
Df Residuals:                   49810   BIC:                        -2.609e+05
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

In [12]:
# --- 7b  H2 validation checks ------------------------------------------------------
T_STAT_THR = 2.5   # "highly significant"

params, tvals, pvals = model_h2.params, model_h2.tvalues, model_h2.pvalues

def _report(name, key):
    b, t, p = params[key], tvals[key], pvals[key]
    print(f'  {name:<12} beta={b:+.4e}  t={t:+.2f}  p={p:.2e}')
    return b, t, p

print(f'R-squared = {model_h2.rsquared:.4f}  (HAC, maxlags={m})\n')
print('Coefficients:')
b1, t1, p1 = _report('STV (b1)', 'stv')
b2, t2, p2 = _report('NOI (b2)', 'noi')
b3, t3, p3 = _report('OBI (b3)', 'obi')

stv_ok = (b1 > 0) and (t1 > T_STAT_THR)
noi_ok = (b2 > 0) and (t2 > T_STAT_THR)
obi_negligible = (abs(t3) < T_STAT_THR) or (p3 > 0.05)

print('\n--- H2 validation ---')
print(f'  [{"PASS" if stv_ok else "FAIL"}] b1 (STV) positive & t>2.5 : '
      f'beta>0={b1 > 0}, t={t1:+.2f}')
print(f'  [{"PASS" if noi_ok else "FAIL"}] b2 (NOI) positive & t>2.5 : '
      f'beta>0={b2 > 0}, t={t2:+.2f}')
print(f'  [{"PASS" if obi_negligible else "FAIL"}] b3 (OBI) negligible (|t|<2.5 or p>0.05) : '
      f'|t|={abs(t3):.2f}, p={p3:.2e}')

if stv_ok and noi_ok and obi_negligible:
    print('\n==> H2 CONFIRMED: contemporaneous price impact driven by STV & NOI; OBI negligible.')
else:
    print('\n==> H2 NOT fully supported on this sample (see per-criterion results above).')

R-squared = 0.0026  (HAC, maxlags=15)

Coefficients:
  STV (b1)     beta=+8.2549e-08  t=+6.90  p=5.19e-12
  NOI (b2)     beta=+8.3963e-08  t=+9.16  p=5.28e-20
  OBI (b3)     beta=-9.6398e-09  t=-10.12  p=4.72e-24

--- H2 validation ---
  [PASS] b1 (STV) positive & t>2.5 : beta>0=True, t=+6.90
  [PASS] b2 (NOI) positive & t>2.5 : beta>0=True, t=+9.16
  [FAIL] b3 (OBI) negligible (|t|<2.5 or p>0.05) : |t|=10.12, p=4.72e-24

==> H2 NOT fully supported on this sample (see per-criterion results above).


## 8  Hypothesis H3 — Price Change Prediction (Research Plan §5.3)

Fit the **predictive** regression

$$\Delta P_{t+1} = \theta_0 + \theta_1 NOI_t + \theta_2 OBI_t + \theta_3 STV_t + \eta_{t+1}$$

The target $\Delta P_{t+1}$ is `delta_p` shifted **back one bin within each session
`[SECURITY, DATE]`** (current features predicting the *next intraday* bin); the last bin of every
session has no forward target and is dropped — so a prediction never reaches across an overnight
gap into the next day. As in H2, $OFI_t$ (`ofi`) is omitted (collinear with $NOI$/$STV$), and we
use **Newey-West HAC** errors with $m = \lfloor 4 (T/100)^{2/9} \rfloor$ recomputed on the
re-aligned sample.

**H3 is validated** if $\theta_1$ (NOI) and $\theta_2$ (OBI) are positive and highly significant
($t > 2.5$), while $\theta_3$ (STV) is small and negative (mean-reverting) or insignificant.

In [13]:
# --- 8a  Forward-align the target and fit the predictive model --------------------
# delta_p_fwd_t = delta_p_{t+1}: shift delta_p BACK by 1 within each SESSION [security, date],
# so the target is the NEXT INTRADAY bin and never the first bin of the following session.
df_pred = (
    df_signals
    .sort(['security', 'date', 'bin_start_time'])
    .with_columns(delta_p_fwd=pl.col('delta_p').shift(-1).over(['security', 'date']))
    .drop_nulls(subset=['delta_p_fwd'])   # drops the last bin of each session (no intraday t+1)
)

reg_df = df_pred.select(['delta_p_fwd', 'noi', 'obi', 'stv']).drop_nulls().to_pandas()

T_pred = len(reg_df)                  # re-aligned sample size
m_pred = newey_west_maxlags(T_pred)   # m = floor(4 * (T/100)**(2/9))
print(f'Aligned observations T = {T_pred:,}')
print(f'Newey-West lag truncation m = {m_pred}')

y = reg_df['delta_p_fwd']
X = sm.add_constant(reg_df[['noi', 'obi', 'stv']])   # OFI omitted (collinear)

model_h3 = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': m_pred})
print(model_h3.summary())

Aligned observations T = 49,334
Newey-West lag truncation m = 15
                            OLS Regression Results                            
Dep. Variable:            delta_p_fwd   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     43.68
Date:                Mon, 08 Jun 2026   Prob (F-statistic):           3.50e-28
Time:                        11:13:10   Log-Likelihood:             1.3247e+05
No. Observations:               49334   AIC:                        -2.649e+05
Df Residuals:                   49330   BIC:                        -2.649e+05
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

In [14]:
# --- 8b  H3 validation checks ------------------------------------------------------
T_STAT_THR = 2.5

params, tvals, pvals = model_h3.params, model_h3.tvalues, model_h3.pvalues

def _report(name, key):
    b, t, p = params[key], tvals[key], pvals[key]
    print(f'  {name:<12} theta={b:+.4e}  t={t:+.2f}  p={p:.2e}')
    return b, t, p

print(f'R-squared = {model_h3.rsquared:.4f}  (HAC, maxlags={m_pred})\n')
print('Coefficients:')
th1, tt1, tp1 = _report('NOI (th1)', 'noi')
th2, tt2, tp2 = _report('OBI (th2)', 'obi')
th3, tt3, tp3 = _report('STV (th3)', 'stv')

noi_ok = (th1 > 0) and (tt1 > T_STAT_THR)
obi_ok = (th2 > 0) and (tt2 > T_STAT_THR)
# STV: small & negative (mean-reverting) OR statistically insignificant.
stv_meanrevert = (th3 < 0) or (abs(tt3) < T_STAT_THR) or (tp3 > 0.05)

print('\n--- H3 validation ---')
print(f'  [{"PASS" if noi_ok else "FAIL"}] th1 (NOI) positive & t>2.5      : '
      f'theta>0={th1 > 0}, t={tt1:+.2f}')
print(f'  [{"PASS" if obi_ok else "FAIL"}] th2 (OBI) positive & t>2.5      : '
      f'theta>0={th2 > 0}, t={tt2:+.2f}')
print(f'  [{"PASS" if stv_meanrevert else "FAIL"}] th3 (STV) negative or insignificant: '
      f'theta={th3:+.2e}, t={tt3:+.2f}, p={tp3:.2e}')

if noi_ok and obi_ok and stv_meanrevert:
    print('\n==> H3 CONFIRMED: NOI & OBI predict next-bin returns; STV is weak/mean-reverting.')
else:
    print('\n==> H3 NOT fully supported on this sample (see per-criterion results above).')

R-squared = 0.0029  (HAC, maxlags=15)

Coefficients:
  NOI (th1)    theta=+1.6696e-09  t=+0.87  p=3.82e-01
  OBI (th2)    theta=+5.6976e-09  t=+7.46  p=8.89e-14
  STV (th3)    theta=+2.2003e-07  t=+5.75  p=8.93e-09

--- H3 validation ---
  [FAIL] th1 (NOI) positive & t>2.5      : theta>0=True, t=+0.87
  [PASS] th2 (OBI) positive & t>2.5      : theta>0=True, t=+7.46
  [FAIL] th3 (STV) negative or insignificant: theta=+2.20e-07, t=+5.75, p=8.93e-09

==> H3 NOT fully supported on this sample (see per-criterion results above).
